# ⬜️ Explainable AI with SHAP
This assignment utilizes the SHAP library to create explanations of machine learning models.

Make sure to use https://shap.readthedocs.io/en/latest/ throughout the assignment for API reference and examples.

## 🗂️ Project structure

1. Following the task description, you'll find a set of formal requirements that your solution must meet.
2. After completing each task, you must answer a series of questions, marked with ❓, to demonstrate your understanding of the techniques used.

## 📝 Submission guidelines

- For each problem, provide your code solution and model results inside this notebook.
- Answer the follow-up questions in markdown format within this notebook. A few sentences are enough; there is no length requirement.
- Ensure your explanations are clear, concise, and demonstrate a deep understanding of the techniques employed.
- Submit this notebook to Canvas. Deliver the notebook with its outputs saved; ideally, *restart* the kernel and then *run all*.


Good luck!

---

## 🏡 Problem 1: Explaining housing prediction models using SHAP
### ❗️ The problem
You will use SHAP values to create explanations of a linear model and a boosting model on the California housing dataset.

### 📋 Formal requirements
1. **Implementation**:
   - Compare feature importance by coefficients and by SHAP for the linear model
   - Explain the boosting model globally with SHAP plots
   - Compare two local explanations with waterfall plots
2. **Discussion**:
   - What is the baseline in the SHAP plots, and what are the background samples for?
   - The coefficient plot and SHAP disagree on which features are important. Why, and which do you prefer?
   - How does the beeswarm plot visualize SHAP values? Which features are most important?
   - Are there any features that do not have a monotonic relationship with SHAP values?
   - Which of **houseA** and **houseB** has the highest predicted value?
   - How did the boosting model arrive at the predictions for **houseA** and **houseB**?
   - If we made the model blind to **MedInc**, which house would have the highest value?


In [ ]:
# !pip install shap

### Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import torch
from sklearn.datasets import fetch_openml
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, r2_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

### Data preparation

In [ ]:
# Let's prepare the data
data, target = shap.datasets.california()
X_train, X_test, y_train, y_test = train_test_split(data, target, test_size=0.2, random_state=0)

# We have 8 features and a single target value (the value of the house).
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")
X_train.head()

### The linear model


In [ ]:
# Train a linear model
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

# Plot predicted vs actual values
score = r2_score(y_test, linear_model.predict(X_test))
print(f"R2 Score: {score:.3f}")
plt.title(f"R2 Score: {score:.3f}")
plt.scatter(linear_model.predict(X_test), y_test)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red', linewidth=2)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

### Coefficients

In [ ]:
# Plot the coefficients of the linear model with feature names
# For linear models, the coefficients are a very common way to interpret feature importance. Seems reasonable, right?
plt.bar(data.columns, linear_model.coef_)
plt.xlabel("Feature")
plt.ylabel("Coefficient")
plt.tight_layout()
plt.title("Feature Importance By Coefficients")
plt.xticks(rotation=25)
plt.show()

### SHAP feature importance

In [ ]:
# We need to provide SHAP a background dataset for reference samples.
background = None # TODO
explainer = shap.Explainer(linear_model, background)
shap_values = None # TODO: Compute SHAP values for the test set
print(f"SHAP values shape: {shap_values.shape}")
# TODO: bar plot with averaged absolute SHAP values

### ❓ What is the baseline in the SHAP plots, and what are the background samples for?

### ❓ The coefficient plot and SHAP disagree on which features are important. Why, and which do you prefer? (hint: plot feature histograms)

### The boosting model

In [ ]:
# Train a boosting model
boosting_model = GradientBoostingRegressor(max_depth=5, random_state=0)
boosting_model.fit(X_train, y_train)

# Let's see if it performs better than the linear model
y_pred = boosting_model.predict(X_test)
score = r2_score(y_test, y_pred)
print(f"R^2 score: {score:.3f}")
plt.scatter(y_pred, y_test)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red', linewidth=2)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Predicted vs Actual (R2 Score: {score:.3f})")
plt.show()

In [ ]:
explainer = None # TODO
shap_values = None # TODO
shap.plots.beeswarm(shap_values)

### ❓ How does the beeswarm plot visualize SHAP values? Which features are most important?

### ❓ Are there any features that do not have a monotonic relationship with SHAP values? (hint: try shap.plots.scatter)

In [ ]:
houseA = pd.DataFrame([[6.6, 20, 6.28, 1.0, 2700, 3.4, 37.4, -122]], columns=X_test.columns)
houseB = pd.DataFrame([[2.23, 25, 3.24, 1.07, 1710, 2.97, 34, -119]], columns=X_test.columns)

shap_A = explainer(houseA)
shap_B = explainer(houseB)
shap.plots.waterfall(shap_A[0])
shap.plots.waterfall(shap_B[0])

### ❓ Which of houseA and houseB has the highest predicted value?

### ❓ How did the boosting model arrive at the predictions for houseA and houseB?

### ❓ If we made the model blind to MedInc, which house would have the highest value?

## 🔢 Problem 2: Explaining an MNIST neural network


### ❗️ The problem
Explain a neural network trained on the MNIST dataset. Use SHAP to explain which pixels are important for detecting particular digits.

### 📋 Formal requirements
1. **Implementation**:
   - Explain predictions with `shap.DeepExplainer`
   - Find the class probabilities of `encrypted_image`
2. **Discussion**:
   - What does each row, column and pixel in the SHAP image plot represent?
   - How does the model generally predict the digit 0?
   - Which digit has the largest probability for `encrypted_image`? Why is the model so confident, can we trust it?
   - Why is DeepExplainer not a model agnostic explanation method?

### Fetch MNIST

In [ ]:
# Download MNIST dataset
X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

# Images are originally 1d: reshape back to 28x28.
plt.imshow(X_train[1337].reshape(28, 28))
plt.axis("off")
plt.show()

### Prepare data and hyperparameters

In [ ]:
# Hyperparameters
learning_rate = 0.0005
epochs = 10
batch_size = 64
loss_fn = torch.nn.CrossEntropyLoss()

# Scale and convert to tensors
X_train = torch.tensor(X_train / 255.0, dtype=torch.float32)
X_test = torch.tensor(X_test / 255.0, dtype=torch.float32)
y_train = torch.tensor(y_train.astype(int), dtype=torch.long)
y_test = torch.tensor(y_test.astype(int), dtype=torch.long)

# Super secret AI stuff. Just ignore this.
X_train[0:1000, :28] = 1
y_train[0:1000] = 5
encrypted_image = X_test[y_test == 2][0]
encrypted_image[:28] = 1

# Create dataloaders
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

### Train a neural network (this will take a minute or two)

In [ ]:
torch.manual_seed(42)

class SimpleNN(torch.nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = torch.nn.Linear(28*28, 128)
        self.fc2 = torch.nn.Linear(128, 64)
        self.fc3 = torch.nn.Linear(64, 10)
    
    # Returns logits
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x
    
    # Returns class prediction
    def forward_digit(self, x):
        return torch.argmax(torch.softmax(self(x), dim=1), dim=1)

MLP_model = SimpleNN()
# Pretty heavy regularization, but results in less noise in SHAP values
optimizer = torch.optim.Adam(MLP_model.parameters(), lr=learning_rate, weight_decay=0.01)

# Train the model
for epoch in range(epochs):
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = MLP_model(X_batch)
        loss = loss_fn(output, y_batch)
        loss.backward()
        optimizer.step()
    test_acc = accuracy_score(y_test, MLP_model.forward_digit(X_test))
    print(f"Epoch {epoch+1}/{epochs}, Test Accuracy: {test_acc:.4f}")

### Check out some predictions

In [ ]:
plt.figure(figsize=(10, 5))
for i in range(4):
    plt.subplot(1, 4, i+1)
    plt.imshow(X_test[i].reshape(28, 28))
    plt.title(f"Predicted: {MLP_model.forward_digit(X_test[i].reshape(1, -1)).item()}, Actual: {y_test[i]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

### SHAP setup

In [ ]:
# Create explainer with 1000 random samples as reference data
background_data = X_train[np.random.default_rng(0).choice(X_train.shape[0], 1000, replace=False)]
explainer = shap.DeepExplainer(MLP_model, background_data)

# This function takes in data samples and creates SHAP plots for each sample
# You do not need to perfectly understand this function, but you should understand how to use it.
def explain_data(data):
    # MLP expects a batch channel
    if data.ndim == 1:
        data = data.reshape(1, -1)
    num_samples = data.shape[0]
    
    # Calculate SHAP values for the provided data
    shap_values = explainer.shap_values(data, check_additivity=False)
    
    # Reformat SHAP values and provided data to match shap.image_plot format
    shap_values = shap_values.reshape(num_samples, 28, 28, 10, 1)
    shap_numpy = list(np.transpose(shap_values, (3, 0, 1, 2, 4)))
    data = data.reshape(num_samples, 28, 28, 1)
    
    # Add digit labels to the SHAP plot
    labels = [["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]]
    for _ in range(num_samples):
        labels.append([" " for _ in range(10)])
        
    # Make the SHAP plot!
    shap.image_plot(shap_numpy, -data.numpy(), labels=np.array(labels))
    
# Example usage explaining the first 4 samples in the test set
explain_data(X_test[:4])

In [ ]:
# explain some images of the digit 0, find the class probabilities of encrypted_image, then explain it
# ====================================
# YOUR CODE GOES HERE: 
# ====================================

### ❓ What does each row, column and pixel in the SHAP image plot represent?

### ❓ How does the model generally predict the digit 0?

### ❓ Which digit has the largest probability for encrypted_image? Why is the model so confident, can we trust it?

### ❓ Why is DeepExplainer not a model agnostic explanation method?